In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
import geopandas as gpd
import scipy.stats as stats
import rasterio
import rioxarray as rxt


%matplotlib inline

## Sampling Parameters from Imsang

In [ ]:
import fiona
print(fiona.__version__)

In [ ]:
# directory
result_dir = r"D:/ForestFire/CBH/result"

In [ ]:
# funciton with log transformation of DBH & Height
def func2(X, a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6):
    H, D, EL, SL, AZ, CD = X
    H_log, D_log = np.log1p(H), np.log1p(D)
    size = (b1 * H_log/D_log)+(b2 * H_log)+(b3 * D_log**2)
    comp = c1 * CD
    site = (d1 * EL)+(d2 * EL**2) + (d3 * SL)+(d4 * SL**2)+(d5 * SL * np.sin(AZ))+(d6 * SL * np.cos(AZ))
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr

In [ ]:
layers = fiona.listlayers(r"G:\CBH\Imsang_merge.gdb")
layers

In [ ]:
# 임상도 불러오기
gdb_dir = r"G:\CBH\Imsang_merge.gdb"
if fiona.listlayers(gdb_dir):
    layer = fiona.listlayers(gdb_dir)
    imsang = gpd.read_file(os.path.join(gdb_dir), layer=layer[0])

In [ ]:
imsang[['DBH(cm)', 'Height(m)', 'CD(%)']] = np.zeros((len(imsang), 3))

In [ ]:
filtered_imsang = imsang[~imsang['KOFTR_GROU'].isin([str(i) for i in [65, 66, 67, 68, 77, 78, 81, 82, 83, 91, 92, 93, 94, 95, 99]])]
print(len(filtered_imsang.index), len(imsang.index))
# imsang_index, koftr_cd, dbh, height, crown density, elevation, slope
result_array = np.zeros((len(filtered_imsang), 7))
result_array[:, :2] = np.vstack((filtered_imsang.index, filtered_imsang['KOFTR_GROU'])).T
len(result_array)

In [ ]:
[if i not in for i in np.unique(imsang.KOFTR_GROU)]

In [ ]:
# 42 -> 40
filtered_imsang['HEIGHT'] = filtered_imsang['HEIGHT'].apply(lambda x: '40' if x == '42' else x)
filtered_imsang['HEIGHT'] = filtered_imsang['HEIGHT'].apply(lambda x: '00' if x == '0' else x)
filtered_imsang['HEIGHT'] = filtered_imsang['HEIGHT'].apply(lambda x: '16' if x == '15' else x)

In [ ]:
# 경급, 임분고, 수관밀도  random 추출 및 적용
dbh_dict = {'0' : [0, 6], '1' : [6, 18], '2' : [18, 30], '3' : [30, 107]}
h_dict = {f"{i*2:02d}": [i*2 - 1, i*2 + 1] for i in range(21)} # f':02d' : 10 미만의 수는 0으로 padding 넣기
cd_dict = {'A' : [0, 50], 'B' : [51, 70], 'C' : [71, 100]}
h_dict['00'] = [0, 1]
h_dict

In [ ]:
filtered_imsang.shape

In [ ]:
# 수관밀도: Gamma 분포 적용
# Given Gamma distribution parameters from the image

# make height samples
alpha = 10.22 
beta = 43.14
loc = -1.56
scale = 76.72
h_samples = stats.beta.rvs(alpha, beta, loc=loc, scale=scale, size=100000)
h_samples = h_samples[h_samples > 0]

# make dbh samples
shape = 1.58  # Shape parameter
loc = 7.56  # Location parameter
scale = 15.68   # Scale parameter
dbh_samples = stats.weibull_min.rvs(shape, loc=loc, scale=scale, size=100000)
dbh_samples = dbh_samples[dbh_samples > 0]

# make cdown density samples
shape = 84.21  # Shape parameter
loc = -111.64  # Location parameter
scale = 2.31   # Scale parameter
cd_samples = stats.gamma.rvs(shape, loc=loc, scale=scale, size=10000)
cd_samples = cd_samples[(cd_samples > 0.) & (cd_samples <= 100.)]

# error 저장
err_idx = []
# sample 추출 후 저장
for idx, data in tqdm(enumerate(filtered_imsang[['DNST_CD', 'DMCLS_CD', 'HEIGHT']].values), total=len(filtered_imsang)):
    try:
        cd, dbh, h = data
        # 수관밀도 추출
        if cd.strip(): # 비어있는 string 제거
            lower, upper = cd_dict[cd]
            filtered_samples = cd_samples[(cd_samples > lower) & (cd_samples <= upper) & (cd_samples > 0.)]
            if len(filtered_samples) > 0:
                result_array[idx, 4] = np.random.choice(filtered_samples)
        # DBH 추출
        if dbh.strip():
            lower, upper = dbh_dict[dbh]
            filtered_samples = dbh_samples[(dbh_samples > lower) & (dbh_samples <= upper) & (dbh_samples > 0.)]
            if len(filtered_samples) > 0:
                result_array[idx, 2] = np.random.choice(filtered_samples)
                
        # Height 추출
        if h.strip():
            lower, upper = h_dict[h]
            filtered_samples = h_samples[(h_samples > lower) & (h_samples <= upper)  & (h_samples > 0.)]
            if len(filtered_samples) > 0:
                result_array[idx, 3] = np.random.choice(filtered_samples)
    except KeyError as e:
        print(e)
        err_idx.append(idx)

In [ ]:
# 단위 변환
cm_to_ft = 0.0328084
m_to_ft = 3.28084
result_array[:, 3] = result_array[:, 2] * cm_to_inch
result_array[:, 3] = result_array[:, 3]  * m_to_ft

In [ ]:
# 수정
data = np.loadtxt(os.path.join(result_dir, 'imsang_params4.txt'), dtype='float')
data[:, 3] = data[:, 3] / cm_to_ft * m_to_ft
np.savetxt(os.path.join(result_dir, 'imsang_params5.txt'), data, fmt='%.4f')

In [ ]:
# result array 임시 저장
np.savetxt(os.path.join(result_dir, 'imsang_params4.txt'), result_array, fmt='%.4f')

In [ ]:
# check the distribution
data = np.loadtxt(os.path.join(result_dir, 'imsang_params4.txt'), dtype='float')
features = ['imsang_index', 'koftr_cd', 'dbh', 'height', 'crown density', 'elevation', 'slope']
# distribution of dbh
fig, ax = plt.subplots(1, 3, figsize=(8, 4))
for i in range(3):
    ax[i].hist(np.clip(data[:, i + 2], 0.1, max(data[:, i])), bins=100)
    ax[i].set_title(f'Histogram : {features[i+2]}')
plt.show()

In [ ]:
len(data[data[:, 3] <= 0. , 3])

In [ ]:
filtered_samples = dbh_samples[(dbh_samples > lower) & (dbh_samples <= upper) & (dbh_samples > 0.)]
if len(filtered_samples) > 0:
    result_array[idx, 4] = np.random.choice(filtered_samples)
else:
    print(f"No valid samples found for index {idx}.")

### DEM에서 elevation, slope 뽑기

In [ ]:
data = np.loadtxt(os.path.join(result_dir, 'imsang_params2.txt'), dtype='float')
data

In [ ]:
filter_imsang_gdf = gpd.GeoDataFrame(filtered_imsang)
len(filter_imsang_gdf)

In [ ]:
dem = r'G:\Digitize\DEM_5M\MergedDEM.tif'
slope = r"G:\Digitize\slope.tif"

In [ ]:
from rasterio.mask import mask
from shapely.geometry import mapping
from rasterio.warp import calculate_default_transform, reproject, Resampling

In [ ]:
# coordinate verification
imsang_crs = gpd.GeoDataFrame(imsang).crs
print(imsang_crs)
with rasterio.open(dem) as src:
    dem_crs = src.crs
    print(dem_crs)
    print(src.bounds)
    print(src.transform)

In [ ]:
with rasterio.open(slope) as src:
    slope_crs = src.crs
    print(slope_crs)
    print(src.bounds)
    print(src.transform)

In [ ]:
# reproject dem to imsang coordination
reprojected_dem = os.path.join(result_dir, 'DEM5M_reprojected.tif')
if imsang_crs == dem_crs:
    print('crs matches')
else:
    with rasterio.open(dem) as src:
        # Calculate transform and new shape
        transform, width, height = calculate_default_transform(
            src.crs, imsang_crs, src.width, src.height, *src.bounds
        )
        
        # Update metadata
        new_meta = src.meta.copy()
        new_meta.update({
            "crs": imsang_crs,
            "transform": transform,
            "width": width,
            "height": height
        })
    
        # Reproject raster
        with rasterio.open(reprojected_dem, "w", **new_meta) as dst:
            for i in range(1, src.count + 1):  # Loop over raster bands
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=imsang_crs,
                    resampling=Resampling.nearest  # Change to Resampling.bilinear if needed
                )
    
    print("Reprojected")

In [ ]:
# reproject slope
reprojected_slope = os.path.join(result_dir, 'Slope_reprojected.tif')
if imsang_crs == slope_crs:
    print('crs matches')
else:
    with rasterio.open(slope) as src:
        # Calculate transform and new shape
        transform, width, height = calculate_default_transform(
            src.crs, imsang_crs, src.width, src.height, *src.bounds
        )
        
        # Update metadata
        new_meta = src.meta.copy()
        new_meta.update({
            "crs": imsang_crs,
            "transform": transform,
            "width": width,
            "height": height
        })
    
        # Reproject raster
        with rasterio.open(reprojected_slope, "w", **new_meta) as dst:
            for i in range(1, src.count + 1):  # Loop over raster bands
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=imsang_crs,
                    resampling=Resampling.nearest  # Change to Resampling.bilinear if needed
                )
    
    print("Reprojected")

In [ ]:
# 임상도에 고도, 경사 추출하기
# Open DEM raster
with rasterio.open(reprojected_dem) as src:
    dem_crs = src.crs
    if imsang_crs != dem_crs:
        print('The coordinate system does not match! Need Reprojection')
    mean_elevations = []
    
    for geom in tqdm(imsang.geometry):
        # Mask raster using the geometry
        if geom == None:
            mean_elevations.append(np.nan)
            continue
        # This function masks raster using geometry with converting shapefile to GeoJSON file  
        out_image, _ = mask(src, [mapping(geom)], crop=True)
        
        # Compute mean, ignoring NaN values
        mean_val = np.nanmean(out_image)
        mean_elevations.append(mean_val)

# Add to GeoDataFrame
filtered_imsang["meanElev"] = mean_elevations

# Open Slope raster
with rasterio.open(reprojected_slope) as src:
    mean_slopes = []
    
    for geom in tqdm(filter_imsang_gdf.geometry):
        # Mask raster using the geometry
        out_image, _ = mask(src, [geom], crop=True)
        
        # Compute mean, ignoring NaN values
        mean_val = np.nanmean(out_image)
        mean_slopes.append(mean_val)

# Add to GeoDataFrame
filtered_imsang["meanSlope"] = mean_slopes

### Rasterize imsang

In [ ]:
imsang.insert(0, 'ID', imsang.index)
imsang.head(5)

In [ ]:
with rasterio.open(reprojected_dem) as src:
    dem_meta = src.meta.copy()
    dem_crs = src.crs
    dem_transform = src.transform
    dem_shape = (src.height, src.width)
print('Read information from DEM raster')

# 임상도 rasterize
shapes1 = [(geom, v1) for geom, v1 in zip(imsang.geometry, imsang.ID)]
shapes2 = [(geom, v2) for geom, v2 in zip(imsang.geometry, imsang.KOFTR_GROU)]
rasterized_imsang1 = rasterio.features.rasterize(
    shapes = shapes1,
    out_shape = dem_shape,
    transform = dem_transform,
    fill=np.nan,
    dtype = rasterio.int32
)
"""rasterized_imsang2 = rasterio.features.rasterize(
    shapes = shapes2,
    out_shape = dem_shape,
    transform = dem_transform,
    fill=-99,
    dtype = rasterio.int32
)
print('Rasterize imsang shapefile')"""

# save the rasterized imsang shape
imsang_raster = os.path.join(result_dir, 'imsang_raster.tif')
dem_meta.update({'count':1, 'dtype': rasterio.int32, 'nodata':-99.})
with rasterio.open(imsang_raster, 'w', **dem_meta) as dst:
    dst.write(rasterized_imsang1, 1)
    # dst.write(rasterized_imsang, 2)

print('Save imsang raster file')

# visuzlie new imsang raster
with rasterio.open(imsang_raster) as src:
    ras_data = src.read(2)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
plt.figure(figsize=(8, 6))
plt.imshow(ras_data, cmap='blue')
plt.colorbar(label='Species')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

### Band Composite

In [ ]:
# add the values as the band
filtered_imsang
with raster.open(imsang_raster) as src:
    arr1 = src.read(1)
    arr2 = src.read(2)

In [ ]:
# call and open the parameters
params_all = np.loadtxt(os.path.join(result_dir, 'imsang_params.txt'), dtype='object')


# EDA

In [ ]:
parent_dir = r"D:/ForestFire/CBH"
df = pd.read_csv(os.path.join(parent_dir, r'data/NFI7-Immok-Filtered3.csv'))
df.info()

In [ ]:
# 수종 분석
nfi_names = df['수종명'].unique()
nfi_imsang = [df.loc[(df['수종명']==name), '침활구분'].unique()[0] for name in nfi_names]
nfi_dict = {i : j for i, j in zip(nfi_names, nfi_imsang)}
# 임상도 수종 및 코드 분류에 따라 수종코드(SID) 부여하기
id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 10, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 30, 61, 62, 63, 64]
name_lst1 = ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', '굴참나무', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', 
             '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', '포플러', ' 벚나무', '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무']
name_lst2 = ['기타침엽수', '기타 참나무류', '기타활엽수']
no_other = [nfi for nfi in nfi_names if nfi in name_lst1]
other_confi = [name for name in nfi_names if (name not in no_other) & (nfi_dict[name] == '침엽수')]
other_deci = [name for name in nfi_names if (name not in no_other) & (nfi_dict[name] == '활엽수')]

In [ ]:
df2 = df[['표본점번호', '수종명', '흉고직경', '수고', '수령', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']]
cm_to_inch = 0.3937
cm_to_ft = 0.0328084
df2['흉고직경'] = df2['흉고직경'].apply(lambda x: x * cm_to_inch)
df2['수고'] = df2['수고'].apply(lambda x: x * cm_to_ft)
df2['지하고'] = df2['지하고'].apply(lambda x: x * cm_to_ft)
df2['해발고(m)'] = df2['해발고(m)'] / 100 # hm로 변환
df2['경사(degree)'] = np.tan(np.radians(df2['경사(degree)'])) # tangent로 변환
df2['방위각(º)'] = np.radians(df2['방위각(º)']) # radian으로 변환
df2['평균수관밀도(%)'] = df2['평균수관밀도(%)'] / 100 # 소수점 자릿수로 변환

In [ ]:
# ['표본점번호', '수종명', '흉고직경', '수고', '수령', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']
df2.columns = ['SampleID', 'Species', 'DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long']
df2.info

In [ ]:
# 수관비율 생성
df2.insert(6, 'CR', (df2['CBH(ft)'] / df2['H(ft)']))
df2.insert(7, 'CH', (df2['H(ft)'] - df2['CBH(ft)']))

In [ ]:
df2.loc[df2['Species'] == '일본잎갈나무', 'Species'] = '낙엽송'

In [ ]:
# 임상도 수종 및 코드 분류에 따라 수종코드(SID) 부여하기
id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 10, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 30, 61, 62, 63, 64]
name_lst = ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '기타침엽수', '상수리나무', '신갈나무', '굴참나무', '기타 참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무',
           '호두나무', '백합나무', '포플러', ' 벚나무', '느티나무', '층층나무', '아까시나무', '기타활엽수', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무']
quercus = ['갈참나무', '떡갈나무', '졸참나무'] # 참나무류
print(len(id_lst), len(name_lst))
s_dict = {j : i for i, j in zip(id_lst, name_lst)}
sid_lst = []
for idx, name in enumerate(df2['Species']):
    if name in s_dict.keys(): sid_lst.append(s_dict[name])
    else:
        if ((df.loc[idx, '침활구분'] == '활엽수') & (df.loc[idx, '수종명'] not in quercus)): sid_lst.append(30)
        elif (df.loc[idx, '침활구분'] == '침엽수'): sid_lst.append(10)
        else: sid_lst.append(34)
len(sid_lst) == df2.shape[0]
if not 'SID' in df2.columns: df2.insert(1, 'SID', sid_lst)
else: df2['SID'] = sid_lst

In [ ]:
# 침활 구분도 추가하기
if not 'Imsang' in df2.columns: df2.insert(3, 'Imsang', df['침활구분'])
else: df2['Imsang'] = df['침활구분']

In [ ]:
if 'Age' in df2.columns: 
    df3 = df2.drop(columns=['Age']).dropna()
df3.info()

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

In [ ]:
# DBH, Height 정규화하기
scaler1 = StandardScaler()
robust1 = RobustScaler()
dbh_scaled = scaler1.fit_transform(df3[['DBH(inch)']])
dbh_scaled2 = robust1.fit_transform(df3[['DBH(inch)']])
dbh_log = np.log1p(df3[['DBH(inch)']])
dbh_exp = np.exp(df3[['DBH(inch)']])
scaler2 = StandardScaler()
robust2 = RobustScaler()
h_scaled = scaler2.fit_transform(df3[['H(ft)']])
h_scaled2 = robust2.fit_transform(df3[['H(ft)']])
h_log = np.log1p(df3[['H(ft)']])
h_exp = np.exp(df3[['H(ft)']])
# 분포 비교하기
fig, ax = plt.subplots(2,5, figsize=(10,5)) # Object-oriented subplot / plt.subplot(1,2,1: State-based)
# plt.grid(color='gray', linestyle='--', linewidth=.5, alpha=.5)
# graphs of DBH
ax[0,0].hist(x=df3['DBH(inch)'], bins=100)
ax[0,0].set_title('Original')
ax[0,1].hist(x=dbh_scaled, bins=100)
ax[0,1].set_title('Standard Scaler')
ax[0,2].hist(x=dbh_scaled2, bins=100)
ax[0,2].set_title('Robust Scaler')
ax[0,3].hist(x=dbh_log, bins=100)
ax[0,3].set_title('Log transform')
ax[0,4].hist(x=dbh_exp, bins=100)
ax[0,4].set_title('Exponential transform')
# graphs of Height
ax[1,0].hist(x=df3['H(ft)'], bins=100)
ax[1,0].set_title('Original')
ax[1,1].hist(x=h_scaled, bins=100)
ax[1,1].set_title('Standard Scaler')
ax[1,2].hist(x=h_scaled2, bins=100)
ax[1,2].set_title('Robust Scaler')
ax[1,3].hist(x=h_log, bins=100)
ax[1,3].set_title('Log Transform')
ax[1,4].hist(x=h_exp, bins=100)
ax[1,4].set_title('Exponential Transform')

fig.text(0.5, 1, "DBH", ha="center", fontsize=14, fontweight="bold")
fig.text(0.5, 0.51, "Height", ha="center", fontsize=14, fontweight="bold")
plt.subplots_adjust(hspace=0.5)


plt.tight_layout()
plt.show()

In [ ]:
def drawPairPlot(df, s_name, opt_save, f_name=None):
    condition = (df['Species'] == s_name)
    df_species = df.loc[condition, ['DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'CR', 'CH', 'Elev(hm)', 'Azimuth(rad)', 'CD(%)']]
    sns.pairplot(df_species, kind='reg', plot_kws={'line_kws' : {'color' : 'orange'}})
    if opt_save: 
        save_dir = r'D:/ForestFire/CBH/fig'
        plt.savefig(os.path.join(save_dir, f_name))

In [ ]:
# 소나무 scatterplot
drawPairPlot(df2, '소나무', opt_save=True, f_name='소나무-Scatter.png')

In [ ]:
# 굴참나무 scatterplot
drawPairPlot(df2, '굴참나무', opt_save=True, f_name='굴참-Scatter.png')

In [ ]:
# 왕벚나무 scatterplot
drawPairPlot(df2, '왕벚나무', opt_save=True, f_name='왕벚-Scatter.png')

In [ ]:
# 고로쇠 scatterplot
drawPairPlot(df2, '고로쇠나무', opt_save=True, f_name='고로쇠-Scatter(30).png')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

In [ ]:
df2.Species

In [ ]:
X_train.shape

In [ ]:
tree_n = '고로쇠나무'
features = ['DBH(inch)', 'CH']
df_pinus = df2[df2['Species'] == tree_n].loc[:, features].reset_index(drop=True)
ratio = .3

df_train = df_pinus.sample(frac=(1 - ratio), replace=False)
df_test = df_pinus.sample(frac=ratio, replace=False)
X_test = df_test[['DBH(inch)']]
y_test = df_test['CH']
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)


# trian and test model
ccf_pinus = LinearRegression()

# Cross-validation for training set
kf = KFold(n_splits=3, shuffle=True, random_state=44)
best_score = -np.inf
cv_scores = []
best_coef = None
best_intercept = None
test_score = None

# perform cross-validation
X = df_train[['DBH(inch)']]
y = df_train['CH']
for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # train model
    ccf_pinus.fit(X_train, y_train)
    score = ccf_pinus.score(X_val, y_val)
    cv_scores.append(score)
    # save the best score
    if score > best_score:
        best_score = score
        best_coefs = ccf_pinus.coef_
        best_intercept = ccf_pinus.intercept_
        # test score
        test_score = ccf_pinus.score(X_test, y_test)
    
print('Best score(r2): ', best_score)
print('Test score(r2): ', test_score)
print('Best coef: ', best_coefs)
print('Best intercept: ', best_intercept)
print(cv_scores)

In [ ]:
from scipy.optimize import minimize # regularization 적용 툴

In [ ]:
# 함수 정의
# Baseline function
def func(X, a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6):
    H, D, EL, SL, AZ, CD = X
    size = (b1 * H/D)+(b2 * H)+(b3 * D**2)
    comp = c1 * CD
    site = (d1 * EL)+(d2 * EL**2) + (d3 * SL)+(d4 * SL**2)+(d5 * SL * np.sin(AZ))+(d6 * SL * np.cos(AZ))
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr
# funciton with log transformation of DBH & Height
def func2(X, a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6):
    H, D, EL, SL, AZ, CD = X
    H_log, D_log = np.log1p(H), np.log1p(D)
    size = (b1 * H_log/D_log)+(b2 * H_log)+(b3 * D_log**2)
    comp = c1 * CD
    site = (d1 * EL)+(d2 * EL**2) + (d3 * SL)+(d4 * SL**2)+(d5 * SL * np.sin(AZ))+(d6 * SL * np.cos(AZ))
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr

# loss function for baseline function
def loss_func(params, lam, X, y):
    y_pred = func(X, *params)
    return np.sum((y - y_pred) ** 2) + lam * np.sum(params**2) # L2 규제 적용

# loss function for func2
def loss_func2(params, lam, X, y):
    y_pred = func2(X, *params)
    return np.sum((y - y_pred) ** 2) + lam * np.sum(params**2) # L2 규제 적용

In [ ]:
# 소나무
sid = '소나무'
lam = 0.1
condition = (df3['Species'] == sid)
df_species = df3.loc[condition, ['H(ft)', 'DBH(inch)','Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)','CR']]
cnt = len(df_species)

if cnt > 10:
    # train-test data split
    X_train, X_test, y_train, y_test = train_test_split(df_species.iloc[:, :-1], df_species['CR'], test_size=.2) #random_state=44)
    
    # create train-test X-array for curve-fit
    X_train = np.array(X_train).T
    X_test = np.array(X_test).T

    # train curvefit
    popt, pcov = curve_fit(func2, np.array(X_train), np.array(y_train))
    result_reg = minimize(loss_func2, x0=popt, args=(lam, X_train, y_train))
    best_params = result_reg.x
    
    y_pred = func2(X_train, *best_params)
    r2_train = r2_score(y_train, y_pred)
    mae_train = mean_absolute_error(y_train, y_pred)
    rmse_train = root_mean_squared_error(y_train, y_pred)
    
    # test curvefit
    y_pred2 = func2(X_test, *best_params)
    r2_test = r2_score(y_test, y_pred2)
    mae_test = mean_absolute_error(y_test, y_pred2)
    rmse_test = root_mean_squared_error(y_test, y_pred2)

In [ ]:
print(r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)

### With Cross Validation

In [ ]:
# Apply CV
# record array 만들기: (species_id, data_count, (coef 11), r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)
rec = np.zeros((len(name_lst), 19))
rec[:, 0] = id_lst
print(rec.shape)
rec
# 'SampleID', 'Species', 'DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long'
lam = 0.1 #0.0 - No regularization, 0.01 - Slight, 0.1 - Moderate, 1 - Overly strong
test_ratio = 0.2
n_fold = 5
tree_lst = [13] # df3['SID'].unique()
for sid in tqdm(tree_lst):
    condition = (df3['SID'] == sid)
    df_species = df3.loc[condition, ['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)','CR']]
    cnt = len(df_species)
    popt = [-999] * 11
    r2_train = None
    mae_train = None
    rmse_train = None
    r2_test = None
    mae_test = None
    rmse_test = None
    
    if cnt > 3:
        # train-test data split
        df_test = df_species.sample(frac=test_ratio, replace=False)
        df_train = df_species.drop(index=df_test.index)
        # create test X, y for curve-fit
        X_test = np.array(df_test.iloc[:, :-1]).T
        y_test = df_test.iloc[:, -1]
        
        # Cross-validation for training set
        kf = KFold(n_splits=n_fold, shuffle=True) # 5th: 66 # 4th: 55 # 3th: 444
        best_score = -np.inf
        cv_scores = []
        best_params = None
        test_score = None
        
        # perform cross-validation
        X = df_train.iloc[:, :-1]
        y = df_train.iloc[:, -1]
        for train_index, val_index in kf.split(X):
            X_train, X_val = X.iloc[train_index], X.iloc[val_index]
            y_train, y_val = y.iloc[train_index], y.iloc[val_index]
            
            # create train-val X-array for curve-fit
            X_train = np.array(X_train).T
            X_val = np.array(X_val).T
            
            # train curvefit
            popt, pcov = curve_fit(func2, np.array(X_train), np.array(y_train))
            
            # optimize the parameters using L2 regularization (minimize funciton)
            result_reg = minimize(loss_func2, x0=popt, args=(lam, X_train, y_train)) # (손실함수, 초기값, 그외 전달인자: lambda값, X, y)
            opt_params = result_reg.x
    
            # predict & evaluate performance with the best params
            y_pred = func2(X_train, *opt_params)
            score = r2_score(y_train, y_pred)
            # save the result of the best score
            if score > best_score:
                best_score = score
                best_params = opt_params
            
        # evaluate with the best-score params
        best_pred = func2(np.array(X).T, *best_params)
        r2_train = r2_score(y, best_pred)
        mae_train = mean_absolute_error(y, best_pred)
        rmse_train = root_mean_squared_error(y, best_pred)

        # test with the best-score params
        y_pred2 = func2(X_test, *best_params)
        r2_test = r2_score(y_test, y_pred2)
        mae_test = mean_absolute_error(y_test, y_pred2)
        rmse_test = root_mean_squared_error(y_test, y_pred2)

    # save the result in the record list
    rec[rec[:, 0] == sid, :] = [sid, cnt] + [r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test] + list(best_params)
    result_dir = r'D:/ForestFire/CBH/result'
    np.savetxt(os.path.join(result_dir, 'CR_Han_result_LogTrans_CV8.4.txt'), rec, delimiter=',',fmt='%.2f', header=','.join(['SID','Count' ,'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te']))

In [ ]:
df_result = pd.DataFrame(rec, columns=['SID','Count' ,'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te'] + [f'Par{i}' for i in range(11)])
df_result.insert(1, 'SName', name_lst)
df_result.to_csv(os.path.join(result_dir, 'CR_Han_result_LogTransform_CV8.4.csv'), encoding='cp949')
df_result2 = df_result[df_result['Count'] >= 30].reset_index(drop=True)
df_result2

In [ ]:
df_result2 = pd.read_csv(os.path.join(result_dir, r'CR_Han_result_LogTransform_CV_FIN.csv'), encoding='cp949')
df_result2.head(3)

In [ ]:
# Set font to support Korean characters
title = "R2 of the equations by species(H&M, with LogTransform & CV)_FIN"
plt.rc('font', family='Malgun Gothic')  # Use 'Malgun Gothic' for Windows or 'AppleGothic' for Mac
plt.rcParams['axes.unicode_minus'] = False  # Ensure minus signs are displayed correctly

# Set figure size
plt.figure(figsize=(12, 6))

# Define bar width and positions
bar_width = 0.4
x = np.arange(len(df_result2["SName"]))

# Create the bars
plt.bar(x - bar_width/2, df_result2["r2_tr"], width=bar_width, label="r2_tr", color="blue", alpha=0.7)
plt.bar(x + bar_width/2, df_result2["r2_te"], width=bar_width, label="r2_te", color="orange", alpha=0.7)

# Add count annotations
for i in range(len(df_result2)):
    plt.text(x[i] - bar_width/2, df_result2["r2_tr"][i] + 0.02, f'{df_result2["r2_tr"][i]:.2f}', ha='center', fontsize=5)
    plt.text(x[i] + bar_width/2, df_result2["r2_te"][i] + 0.02, f'{df_result2["r2_te"][i]:.2f}', ha='center', fontsize=5)

# Labels and title
plt.xlabel("SName", fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title(title, fontsize=14)
plt.xticks(ticks=x, labels=df_result2["SName"], rotation=45, ha="right")
plt.legend()

# Show plot
plt.tight_layout()
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, f'{title}.png'))
plt.show()

### No Cross Validation

In [ ]:
# No CV
# 'SampleID', 'Species', 'DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long'
lam = 0.1 #0.0 - No regularization, 0.01 - Slight, 0.1 - Moderate, 1 - Overly strong
n_fold = 5
for sid in tqdm(df3['SID'].unique()):
    condition = (df3['SID'] == sid)
    df_species = df3.loc[condition, ['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)','CR']]
    cnt = len(df_species)
    popt = [-999] * 11
    r2_train = None
    mae_train = None
    rmse_train = None
    r2_test = None
    mae_test = None
    rmse_test = None
    
    if cnt > 3:
        # train-test data split
        X_train, X_test, y_train, y_test = train_test_split(df_species.iloc[:, :-1], df_species['CR'], test_size=.2, random_state=44)
        
        # create train-test X-array for curve-fit
        X_train = np.array(X_train).T
        X_test = np.array(X_test).T
    
        # train curvefit
        popt, pcov = curve_fit(func2, np.array(X_train), np.array(y_train))
        # optimize the parameters using L2 regularization (minimize funciton)
        result_reg = minimize(loss_func2, x0=popt, args=(lam, X_train, y_train)) # (손실함수, 초기값, 그외 전달인자: lambda값, X, y)
        best_params = result_reg.x

        # predict & evaluate performance with the best params
        y_pred = func2(X_train, *best_params)
        r2_train = r2_score(y_train, y_pred)
        mae_train = mean_absolute_error(y_train, y_pred)
        rmse_train = root_mean_squared_error(y_train, y_pred)
        
        # test with the best params
        y_pred2 = func2(X_test, *best_params)
        r2_test = r2_score(y_test, y_pred2)
        mae_test = mean_absolute_error(y_test, y_pred2)
        rmse_test = root_mean_squared_error(y_test, y_pred2)

    # save the result in the record list
    rec[rec[:, 0] == sid, :] = [sid, cnt] + [r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test] + list(best_params)
    result_dir = r'D:/ForestFire/CBH/result'
    np.savetxt(os.path.join(result_dir, 'CR_Han_result_LogTrans.txt'), rec, delimiter=',',fmt='%.2f', header=','.join(['SID','Count' ,'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te']))

### 누락된 나무 확인

In [ ]:
no_species = ['낙엽송', '편백나무', '가문비나무', '비자나무', '호두나무', '포플러', '벚나무', '가시나무', '녹나무']
df2[df2['Species'].isin(no_species)]

### 다중공선성 확인

In [ ]:
result_dir

In [ ]:
data = np.loadtxt(os.path.join(result_dir, 'CR_Han_result_LogTransform_CV_FIN.csv'), skiprows=1, dtype='object', delimiter=',')
names = data[:, 1]
mask = np.ones(data.shape[1], dtype=bool)
mask[1] = False
data2 = data[:, mask].astype('float')

In [ ]:
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.width", 200)         # Set display width
pd.set_option("display.max_colwidth", 100)  # Set max width for individual cells

In [ ]:
# print the pcov
track = 0
param_name = ['a', 'b1', 'b2', 'b3', 'c1', 'd1', 'd2', 'd3', 'd4', 'd5', 'd6']
file_path = os.path.join(result_dir, 'covariance.xlsx')
if not os.path.exists(file_path):
    # Create an empty DataFrame and save it to the file
    with pd.ExcelWriter(file_path, mode='w', engine='openpyxl') as writer:
        pd.DataFrame().to_excel(writer, sheet_name="Placeholder")
for sid in tqdm(df3['SID'].unique()):
    if data2[(data2[:, 0]==sid), 1] > 10:
        condition = (df3['SID'] == sid)
        df_species = df3.loc[condition, ['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)','CR']]
        X = np.array(df_species.iloc[:, :-1]).T
        y = df_species.iloc[:, -1]
        popt, pcov = curve_fit(func2, X, y, p0=data2[track, -11:])
        df_cov = pd.DataFrame(data=pcov, columns = param_name, index = param_name)
        with pd.ExcelWriter(file_path, mode='a', engine='openpyxl', if_sheet_exists='new') as writer:
            df_cov.to_excel(writer, sheet_name=f'Sheet{sid}', index=True)
        print(f"Covariance matrix of {names[track]}") 
        print(df_cov)
        track += 1